In [31]:
import numpy as np
from pathlib import Path

In [32]:
data_dir = "./initial_data"

In [33]:
def load_function(function_id, data_dir=data_dir):
    path = Path(data_dir) / f"function_{function_id}"

    X = np.load(path / "initial_inputs.npy")
    y = np.load(path / "initial_outputs.npy")

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)

    if len(X) != len(y):
        raise ValueError("X and y contain different numbers of observations.")

    if np.any(X < 0) or np.any(X > 1):
        raise ValueError("Inputs outside [0, 1].")

    return X, y

In [34]:
X, y = load_function(1)

In [35]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    RBF
)

def fit_gp(X, y):
    scaler = StandardScaler()

    kernel = (
        ConstantKernel(1.0,(1e-3, 1e3))
        * RBF(
            length_scale=[1.0] * X.shape[1],
            length_scale_bounds=(1e-5, 1e5)
        )
    )

    X_scaled = scaler.fit_transform(X)

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42,
    )

    model.fit(X_scaled, y)

    return model

In [36]:
from scipy.stats import norm


def expected_improvement(X_candidates, model, y_best):
    mean, std = model.predict(
        X_candidates,
        return_std=True
    )

    std = np.maximum(std, 1e-12)

    improvement = y_best - mean

    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        + std * norm.pdf(z)
    )

    return ei

In [37]:
def generate_candidates(n_candidates, dimension, rng):
    return rng.uniform(
        0,
        1,
        size=(n_candidates, dimension)
    )

In [38]:
def propose_next(
    X,
    y,
    maximise=False,
    n_candidates=100_000,
    seed=42,
):

    # Convert maximisation to minimisation
    target = -y if maximise else y

    model = fit_gp(X, target)

    candidates = generate_candidates(
        n_candidates=n_candidates,
        dimension=X.shape[1],
        rng=np.random.default_rng(seed),
    )

    y_best = np.min(target)

    ei = expected_improvement(
        candidates,
        model,
        y_best,
    )

    best_idx = np.argmax(ei)

    return candidates[best_idx], model

In [39]:
def format_query(x):
    return "-".join(
        f"{value:.6f}"
        for value in x
    )

In [40]:
for i in range(1, 9):
    X, y = load_function(i)

    x_next, model = propose_next(
        X,
        y,
        maximise=False,
    )

    best_idx = y.argmin()
    best_x = X[best_idx]
    best_y = y[best_idx]

    query = format_query(x_next)

    print("\n====================================")
    print(f"FUNCTION {i}")
    print("====================================")
    print("Observations:        ", X.shape[0])
    print("Dimensions:          ", X.shape[1])
    print("Current best x:      ", best_x)
    print("Current best y:      ", f"{best_y:.6f}")
    print("Proposed x:")
    print(f"        {x_next}")
    print("Portal query:")
    print(f"        {query}")
    


FUNCTION 1
Observations:         10
Dimensions:           2
Current best x:       [0.65011406 0.68152635]
Current best y:       -0.003606
Proposed x:
        [0.38718515 0.97161803]
Portal query:
        0.387185-0.971618

FUNCTION 2
Observations:         10
Dimensions:           2
Current best x:       [0.14269907 0.34900513]
Current best y:       -0.065624
Proposed x:
        [4.50627487e-05 3.06208459e-03]
Portal query:
        0.000045-0.003062

FUNCTION 3
Observations:         15
Dimensions:           3
Current best x:       [0.15183663 0.43999062 0.99088187]
Current best y:       -0.398926
Proposed x:
        [0.00619839 0.98457862 0.98668269]
Portal query:
        0.006198-0.984579-0.986683

FUNCTION 4
Observations:         30
Dimensions:           4
Current best x:       [0.94838936 0.89451301 0.85163782 0.55219629]
Current best y:       -32.625660
Proposed x:
        [0.96314056 0.99444099 0.8016244  0.99928591]
Portal query:
        0.963141-0.994441-0.801624-0.999286

FUNCT